In [ ]:
# ---------------------------------------------------------------------------
# C15-c(PC,SR,S,IM) vs. C15-c(PC,SR,S): does adding immunity change the fit?
#
# Reconstructs img/reviewer_peak_incidence_diff_im_vs_noim_map_kde.png.
# Input: data/statistics/city_fe_model_fits/city_predicted_incidence.csv,
# produced by `Rscript 01_model_fit/fit_city_fe_models.R`.
#
# Three panels:
#   1. Map of the 2023/24 peak-month fitted incidence difference, IM − no-IM.
#   2. KDE by region of the same difference (log10(diff + 1)).
#   3. KDE by region of the RMSE (log1p incidence) difference, IM − no-IM,
#      over the WHOLE time series, not just the 2023/24 season.
# ---------------------------------------------------------------------------
import pandas as pd
import numpy as np
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap

city_fits = pd.read_csv(
    'data/statistics/city_fe_model_fits/city_predicted_incidence.csv',
    parse_dates=['date_first_symptoms']
)

season = city_fits[
    (city_fits['date_first_symptoms'] >= '2023-08-01') &
    (city_fits['date_first_symptoms'] <= '2024-07-01')
]

fitted_wide = season.pivot_table(
    index=['city_residency', 'date_first_symptoms'],
    columns='model_name',
    values='fitted_incidence_per100k'
).reset_index()

# Peak month defined by OBSERVED incidence (the real epidemiological peak),
# same convention as peak_df in ReviewerResponse_ImmunityR2ByRegion.ipynb —
# map + middle panel stay on the 2023/24 season only.
observed = season.drop_duplicates(['city_residency', 'date_first_symptoms'])[
    ['city_residency', 'region', 'date_first_symptoms', 'observed_incidence_per100k']
]
peak_dates = observed.loc[observed.groupby('city_residency')['observed_incidence_per100k'].idxmax()]
peak_dates = peak_dates[peak_dates['observed_incidence_per100k'] > 0]

peak_fitted = peak_dates.merge(fitted_wide, on=['city_residency', 'date_first_symptoms'], how='left')
peak_fitted['diff_im_noim'] = peak_fitted['C15-c(PC,SR,S,IM)'] - peak_fitted['C15-c(PC,SR,S)']
peak_fitted['log_diff'] = np.log10(peak_fitted['diff_im_noim'] + 1)
peak_fitted.loc[peak_fitted['region'] == 'Middle-West', 'region'] = 'Centre-West'

# ---------------------------------------------------------------------------
# RMSE (log1p incidence, observed vs. fitted) per city per model, over the
# WHOLE time series in city_fits (not restricted to the 2023/24 season).
# ---------------------------------------------------------------------------
def rmse(x, y):
    return np.sqrt(np.mean((x - y) ** 2))

rmse_by_city_model = (
    city_fits.groupby(['city_residency', 'model_name'])
    .apply(lambda d: rmse(
        np.log1p(d['observed_incidence_per100k']), np.log1p(d['fitted_incidence_per100k'])
    ), include_groups=False)
    .reset_index(name='rmse')
)

rmse_wide = rmse_by_city_model.pivot_table(
    index='city_residency', columns='model_name', values='rmse'
).reset_index()
rmse_wide['diff_rmse'] = rmse_wide['C15-c(PC,SR,S,IM)'] - rmse_wide['C15-c(PC,SR,S)']

city_region = city_fits.drop_duplicates('city_residency')[['city_residency', 'region']]
rmse_wide = rmse_wide.merge(city_region, on='city_residency', how='left')
rmse_wide.loc[rmse_wide['region'] == 'Middle-West', 'region'] = 'Centre-West'

gdf = gpd.read_file('~/shapefiles/municipios/municipios.shp')
gdf['city_name'] = gdf['SIGLA_UF'] + '_' + gdf['city_name']
gdf = gdf.set_index('city_name').join(peak_fitted.set_index('city_residency')[['diff_im_noim']])

region_shp = gpd.read_file('~/shapefiles/BR_UF_2022/BR_UF_2022.shp')
world = gpd.read_file('~/shapefiles/ne_50m_admin_0_countries/ne_50m_admin_0_countries.shp')

abs_scale = np.nanpercentile(np.abs(gdf['diff_im_noim'].dropna()), 90)
norm = TwoSlopeNorm(vmin=-abs_scale, vcenter=0, vmax=abs_scale)
cmap = LinearSegmentedColormap.from_list('RdWhGn', ['#b2182b', '#f7f7f7', '#1a9850'])

fig, axes = plt.subplots(1, 3, figsize=(26, 8), gridspec_kw={'width_ratios': [1.3, 1, 1]})

# --- Left: map of the 2023/24 peak-incidence difference ---
ax = axes[0]
world.plot(ax=ax, color='lightgrey', edgecolor='black', lw=0.5)
gdf.plot(
    ax=ax, column='diff_im_noim', cmap=cmap, norm=norm, legend=True,
    legend_kwds={'label': 'Peak incidence difference, IM − no-IM (cases per 100k)', 'shrink': 0.6, 'extend': 'both'},
    missing_kwds={'color': 'whitesmoke'}
)
region_shp.plot(ax=ax, color='none', edgecolor='black', lw=0.8)
ax.set_xlim(-75, -33)
ax.set_ylim(-34, 6)
ax.set_title('2023/24 peak incidence: C15-c(PC,SR,S,IM) − C15-c(PC,SR,S)')

# --- Middle: KDE of the peak-incidence difference (not normalized), one curve per region ---
ax = axes[1]
sns.kdeplot(
    data=peak_fitted,
    x='log_diff',
    hue='region',
    common_norm=False,
    fill=False,
    lw=2,
    ax=ax
)
ax.axvline(0, color='k', ls='--')
ax.set_xlabel('log(Peak incidence difference), IM − no-IM (cases per 100k)')
ax.set_ylabel('Density')
ax.set_title('Peak incidence: distribution by region')

# --- Right: KDE of the RMSE difference (whole time series), one curve per region ---
ax = axes[2]
sns.kdeplot(
    data=rmse_wide,
    x='diff_rmse',
    hue='region',
    common_norm=False,
    fill=False,
    lw=2,
    ax=ax
)
ax.axvline(0, color='k', ls='--')
ax.set_xlabel('RMSE (log scale) difference, IM − no-IM — whole time series')
ax.set_ylabel('Density')
ax.set_title('Fit error (RMSE, all years): distribution by region')

plt.tight_layout()

n_missing = gdf['diff_im_noim'].isna().sum()
print(f"{n_missing} municipalities have no value on the map (outside the model fit, or zero observed peak all season).")

plt.savefig('img/reviewer_peak_incidence_diff_im_vs_noim_map_kde.png', bbox_inches='tight', dpi=300)
plt.savefig('img/reviewer_peak_incidence_diff_im_vs_noim_map_kde.pdf', bbox_inches='tight', dpi=300)
plt.show()

/apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


270 municipalities have no value on the map (outside the model fit, or zero observed peak all season).
